# GNN Bipartite Tertile Classification

Same setup as `gnn_bipartite_classification.ipynb` but **3-class tertile labels** instead of a binary median split.

- Class 0 = bottom third of next-quarter outcome
- Class 1 = middle third
- Class 2 = top third

Stocks are bucketed by next-quarter `log_return` terciles; investors by next-quarter profitability terciles. Random-chance accuracy is 1/3 (0.333) — anything meaningfully above that is signal.

## Setup

### torch instalttons to work with cuda

In [ ]:
#  run it if you installed un-supported versions
#  pip uninstall -y torch torchvision torchaudio torch_geometric torch_scatter torch_sparse torch_cluster torch_spline_conv

In [ ]:
# this is the version who worked for me
# pip install torch --index-url https://download.pytorch.org/whl/cu124
# pip install torch_geometric

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("cuda build:", torch.version.cuda)



### imports and configure vars

In [24]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, GATConv

In [25]:
# DEFINE ROOT
def project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, cwd.parent, cwd.parent.parent]:
        if (p / "ETL").is_dir():
            return p
    return cwd.parent.parent

ROOT = project_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

WindowsPath('C:/Users/potda/Daniel/BGU/Year_D/סמסטר ז/Final_Project/Social-Network-Stock-Market')

In [26]:
warnings.filterwarnings("ignore", category=UserWarning)

for env_path in [Path.cwd() / ".env", Path.cwd().parent / ".env", Path.cwd().parent.parent / ".env"]:
    if env_path.exists():
        load_dotenv(env_path)
        break

# import the names of tables in postgress database
from ETL.gnn_db_pipeline.config import (
    TARGET_DB,
    TGT_CHANGED_HOLDINGS,
    TGT_STOCKS_RETURN,
    TGT_CIK_AUM,
    TGT_NORMALIZED_HOLDINGS,
)

# extend regular postgress handler
from ETL.gnn_db_pipeline.db_connector import ConfigurablePostgresHandler


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)
torch.manual_seed(0); np.random.seed(0)

handler = ConfigurablePostgresHandler(TARGET_DB)
handler.connect()
print("Connected to", TARGET_DB, "| device", DEVICE)

2026-04-26 01:40:16 - ETL_Pipeline - INFO - Connected to PostgreSQL: postgres@127.0.0.1:5432/13FGNN


Connected to 13FGNN | device cuda


### Configuration of the GNN (Graph Neural Network) , and Global Vars

In [27]:
CUSIP_FIN_TABLE = "cusip_financial_data"
STOCK_FEATURE_COLS = [
    "diluted_eps", "roe", "ev_ebitda", "pe_ratio", "price_to_sales",
    "price_to_book", "debt_to_equity", "dividend_yield", "fcf_per_share",
    "log_return",
]

# there are 3 options for edges
# EDGES_COLUMN_NAME = "change_in_adjusted_weight"
# EDGES_COLUMN_NAME = "change_in_shares"
EDGES_COLUMN_NAME = "change_in_weight"


In [29]:
YEAR, QUARTER = 2019, 3

NUM_CLASSES = 3       # tertiles
HIDDEN_DIM = 64       # bumped from 32 — 3 classes need a bit more capacity
NUM_LAYERS = 2
EPOCHS = 150
LR = 8e-4
DROPOUT = 0.5
MASK_FRAC = 0.20
GAT_HEADS = 4

## Data loaders

In [28]:
def next_year_quarter(year: int, quarter: int):
    return (year + 1, 1) if quarter == 4 else (year, quarter + 1)

# load edges from the the table changed_holdings
def load_edges(year, quarter , edges_col_name):
    return handler.query(f"""
        SELECT cik, cusip, {edges_col_name} AS w
        FROM {TGT_CHANGED_HOLDINGS}
        WHERE year = %s AND quarter = %s AND {edges_col_name} IS NOT NULL
    """, (year, quarter))

# load returns from stocks_return
def load_returns(year, quarter):
    return handler.query(f"""
        SELECT cusip, log_return FROM {TGT_STOCKS_RETURN}
        WHERE year = %s AND quarter = %s AND log_return IS NOT NULL
    """, (year, quarter))

# load aum from cik_aum
def load_aum(year, quarter):
    return handler.query(f"""
        SELECT cik, total AS aum FROM {TGT_CIK_AUM}
        WHERE year = %s AND quarter = %s AND total > 0
    """, (year, quarter))

# load stocks financial features from cusip_financial_data
def load_stock_features(year, quarter) -> pd.DataFrame:
    try:
        fin = handler.query(f"""
            SELECT * FROM {CUSIP_FIN_TABLE}
            WHERE year = %s AND quarter = %s
        """, (year, quarter))
    except Exception as e:
        print(f"  [warn] {CUSIP_FIN_TABLE} not available ({e.__class__.__name__}); zero-filling features")
        fin = pd.DataFrame(columns=["cusip"] + STOCK_FEATURE_COLS)
    rets = load_returns(year, quarter)
    df = fin.merge(rets, on="cusip", how="outer")
    for c in STOCK_FEATURE_COLS:
        if c not in df.columns:
            df[c] = 0.0
    return df[["cusip"] + STOCK_FEATURE_COLS]


def investor_profitability(year, quarter) -> pd.Series:
    ny, nq = next_year_quarter(year, quarter)

    # loading the data about qaurter
    h = handler.query(f"""
        SELECT cik, cusip, weight FROM {TGT_NORMALIZED_HOLDINGS}
        WHERE year = %s AND quarter = %s
    """, (year, quarter))

    #  load the returns
    r = load_returns(ny, nq)

    # we merge the raw data for each cusip the returns of this cik with the duture returns this is our y to predict
    m = h.merge(r, on="cusip", how="inner")

    # calculate the return weighted
    m["contrib"] = m["weight"] * m["log_return"]
    return m.groupby("cik")["contrib"].sum().rename("profitability")

## Tertile labeller

`qcut` into 3 equal-frequency buckets. NaN values → −1 (masked out of loss). Ties get spread by `duplicates='drop'` fallback to numeric thresholds.

In [30]:
def tertile_labels(values: pd.Series) -> pd.Series:
    """Return integer labels in {0, 1, 2} via tertile cut. NaN → -1."""
    s = values.astype(float)
    out = pd.Series(-1, index=s.index, dtype=np.int64)
    valid = s.dropna()
    if valid.empty:
        return out
    try:
        cats = pd.qcut(valid, q=3, labels=[0, 1, 2])
    except ValueError:
        # Too many ties — fall back to numpy quantiles + searchsorted
        q1, q2 = np.quantile(valid, [1/3, 2/3])
        cats = pd.cut(valid, bins=[-np.inf, q1, q2, np.inf], labels=[0, 1, 2], include_lowest=True)
    out.loc[valid.index] = cats.astype(np.int64)
    return out

## Graph builder

we build a graph, for quarter,year,and the edges are druven from the name of the EDGE_COLUMN_NAME

In [31]:
def zscore(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        v = out[c].astype(float)
        v = v.replace([np.inf, -np.inf], np.nan).fillna(v.median() if v.notna().any() else 0.0)
        sd = v.std()
        out[c] = (v - v.mean()) / sd if sd > 0 else 0.0
    return out



In [32]:
# ---- 1) Investor (CIK) feature table ---------------------------------------
def _build_cik_features(edges, year, quarter):
    """One row per CIK with [log_aum, n_holdings, profitability], z-scored.
    Uses past-quarter profitability as a feature (no leakage)."""
    aum = load_aum(year, quarter)

    # past_prof = profitability(t-1) → uses returns at t, all observable at decision time t.
    py, pq = (year - 1, 4) if quarter == 1 else (year, quarter - 1)
    try:
        past_prof = investor_profitability(py, pq).reset_index()
    except Exception:
        past_prof = pd.DataFrame(columns=["cik", "profitability"])

    # n_holdings = number of cusips this CIK appears in, in the Δ-graph
    cik_nh = edges.groupby("cik").size().rename("n_holdings").reset_index()

    cik_df = (aum.merge(cik_nh, on="cik", how="outer")
                   .merge(past_prof, on="cik", how="left"))

    # Impute and transform
    cik_df["aum"] = cik_df["aum"].fillna(
        cik_df["aum"].median() if cik_df["aum"].notna().any() else 0.0)
    cik_df["log_aum"] = np.log(cik_df["aum"].clip(lower=1.0))
    cik_df["n_holdings"] = cik_df["n_holdings"].fillna(0)
    cik_df["profitability"] = cik_df["profitability"].fillna(0.0)

    CIK_FEATS = ["log_aum", "n_holdings", "profitability"]
    return zscore(cik_df, CIK_FEATS), CIK_FEATS

# ---- 2) Stock (CUSIP) feature table ----------------------------------------
def _build_stock_features(year, quarter):
    """One row per cusip with all STOCK_FEATURE_COLS, z-scored."""
    stock_df = load_stock_features(year, quarter)
    return zscore(stock_df, STOCK_FEATURE_COLS)

# ---- 3) Forward-looking outcomes used as labels ----------------------------
def _load_label_sources(year, quarter):
    """Returns at t+1 (per cusip) and investor profitability of t portfolio
    realized over t+1 (per cik). These are the prediction targets."""
    ny, nq = next_year_quarter(year, quarter)
    r_next = load_returns(ny, nq).set_index("cusip")["log_return"]
    prof_next = investor_profitability(year, quarter)
    return r_next, prof_next


# ---- 4) Build node feature matrix X with CIKs first, CUSIPs second ---------
def _assemble_node_matrix(edges, cik_df, stock_df, CIK_FEATS):
    """Stacks CIK rows on top of CUSIP rows, padding each side with zeros so
    every node ends up with the same feature dim (paper trick)."""
    cik_ids = pd.Index(edges["cik"].unique())
    cusip_ids = pd.Index(edges["cusip"].unique())

    cik_df = cik_df.set_index("cik").reindex(cik_ids).fillna(0.0)
    stock_df = stock_df.set_index("cusip").reindex(cusip_ids).fillna(0.0)

    F_cik = cik_df[CIK_FEATS].to_numpy()
    F_stk = stock_df[STOCK_FEATURE_COLS].to_numpy()
    Fdim = F_cik.shape[1] + F_stk.shape[1]

    X = np.zeros((len(cik_ids) + len(cusip_ids), Fdim), dtype=np.float32)
    X[:len(cik_ids), :F_cik.shape[1]] = F_cik   # CIK rows: only CIK features filled
    X[len(cik_ids):, F_cik.shape[1]:] = F_stk   # CUSIP rows: only stock features filled

    cik_pos   = {c: i for i, c in enumerate(cik_ids)}
    cusip_pos = {c: i + len(cik_ids) for i, c in enumerate(cusip_ids)}
    return X, cik_ids, cusip_ids, cik_pos, cusip_pos


# ---- 5) Build undirected edge_index + edge_weight --------------------------
def _assemble_edges(edges, cik_pos, cusip_pos):
    """Each (cik, cusip) edge added in both directions for an undirected graph.
    edge_weight carries the change_in_weight."""
    src = edges["cik"].map(cik_pos).to_numpy()
    dst = edges["cusip"].map(cusip_pos).to_numpy()
    edge_index = np.stack(
        [np.concatenate([src, dst]), np.concatenate([dst, src])], axis=0)
    edge_weight = np.concatenate(
        [edges["w"].to_numpy(), edges["w"].to_numpy()]).astype(np.float32)
    return edge_index, edge_weight


# ---- 6) Assign tertile labels (3 classes, -1 = unknown) --------------------
def _assign_labels(num_nodes, cusip_pos, cik_pos, r_next, prof_next):
    """Stocks → bottom/mid/top tertile of next-quarter log_return.
    Investors → bottom/mid/top tertile of next-quarter profitability."""
    y = np.full(num_nodes, -1, dtype=np.int64)
    if not r_next.empty:
        stk_lbl = tertile_labels(r_next)
        for cusip, idx in cusip_pos.items():
            v = stk_lbl.get(cusip, -1)
            if v >= 0:
                y[idx] = int(v)
    if not prof_next.empty:
        inv_lbl = tertile_labels(prof_next)
        for cik, idx in cik_pos.items():
            v = inv_lbl.get(cik, -1)
            if v >= 0:
                y[idx] = int(v)
    return y


# ---- 7) Orchestrator -------------------------------------------------------
def build_graph(year: int, quarter: int, edges_col_name):
    """Build one (year, quarter) Δ-graph as a torch_geometric Data object.

    Returns (data, meta).
      data.x            node features (CIKs first, then CUSIPs)
      data.edge_index   undirected edges, both directions
      data.edge_weight  change_in_weight per edge
      data.y            tertile labels (0/1/2), -1 for unlabeled
      data.is_cik       boolean mask: True for CIK nodes
      data.has_label    boolean mask: True for nodes with y >= 0
    """
    # 1. Load Δ-edges for this quarter; bail if empty.
    edges = load_edges(year, quarter, edges_col_name)
    if edges.empty:
        raise RuntimeError(f"no Δ-edges for {year} Q{quarter}")

    # 2. Build feature tables (CIK + CUSIP).
    cik_df, CIK_FEATS = _build_cik_features(edges, year, quarter)
    stock_df = _build_stock_features(year, quarter)

    # 3. Pull forward-looking outcomes that become labels.
    r_next, prof_next = _load_label_sources(year, quarter)

    # 4. Stack into a single node feature matrix.
    X, cik_ids, cusip_ids, cik_pos, cusip_pos = _assemble_node_matrix(
        edges, cik_df, stock_df, CIK_FEATS)

    # 5. Build edges.
    edge_index, edge_weight = _assemble_edges(edges, cik_pos, cusip_pos)

    # 6. Assign labels.
    y = _assign_labels(X.shape[0], cusip_pos, cik_pos, r_next, prof_next)

    # 7. Wrap in a PyG Data object with helper masks.
    data = Data(
        x=torch.tensor(X),
        edge_index=torch.tensor(edge_index, dtype=torch.long),
        edge_weight=torch.tensor(edge_weight),
        y=torch.tensor(y),
    )
    data.is_cik = torch.zeros(X.shape[0], dtype=torch.bool)
    data.is_cik[:len(cik_ids)] = True
    data.has_label = data.y >= 0

    meta = {
       "cik_ids": cik_ids, "cusip_ids": cusip_ids,
        "n_cik": len(cik_ids), "n_cusip": len(cusip_ids),
    }
    return data, meta


In [33]:
#for further exploration
edges = load_edges(2016, 3, EDGES_COLUMN_NAME)
cik_df, _ = _build_cik_features(edges, 2016, 3)
stock_df = _build_stock_features(2016, 3)  # inspect stock features
r_next, prof_next = _load_label_sources(2016, 3)
                    # check label distribution

In [34]:
# we show how the df look like
cik_df.head() 

,cik,aum,n_holdings,profitability,log_aum
0,0000002230,9.963093e+08,-0.025487,-0.022514,0.548584
1,0000003520,1.182860e+10,0.681181,0.168619,1.599754
2,0000005272,4.593959e+10,6.067901,-0.995966,2.176193
3,0000007195,1.665470e+08,-0.367172,0.103085,-0.211379
4,0000007789,9.161325e+08,-0.048784,-0.194859,0.512940


In [35]:
# we show hoe the stock df look like
stock_df.head()

,cusip,diluted_eps,roe,ev_ebitda,pe_ratio,price_to_sales,price_to_book,debt_to_equity,dividend_yield,fcf_per_share,log_return
0,000307108,0.017986,0.026824,0.018006,-0.004884,-0.019438,-0.019231,-0.017878,-0.211567,0.017963,-1.679820
1,000360206,0.017986,0.045973,0.018172,-0.017650,-0.019418,-0.015134,-0.017951,-0.154419,0.017963,-0.135085
2,000361105,0.017986,0.032340,0.018049,-0.017471,-0.019448,-0.020413,-0.017948,-0.141880,0.017963,1.081581
3,00081T108,0.017986,0.036560,0.017984,-0.018618,-0.019449,-0.020607,-0.017883,-0.211567,0.017963,-0.696422
4,000868109,0.017986,0.034807,0.017837,-0.018392,-0.019432,-0.020608,-0.017894,0.074609,0.017963,-0.048482


In [13]:
#  data for rest of the model
data, meta = build_graph(YEAR, QUARTER,EDGES_COLUMN_NAME)
print(f"nodes: {data.num_nodes:,}  (CIKs {meta['n_cik']:,}, CUSIPs {meta['n_cusip']:,})")
print(f"edges: {data.num_edges:,}  (undirected, doubled)")
print(f"feat dim: {data.num_node_features}")
labeled = data.has_label
print(f"labeled nodes: {int(labeled.sum()):,}")
print("class distribution (labeled):",
      {int(c): int((data.y[labeled] == c).sum()) for c in [0, 1, 2]})

nodes: 7,778  (CIKs 4,726, CUSIPs 3,052)
edges: 1,793,956  (undirected, doubled)
feat dim: 13
labeled nodes: 7,718
class distribution (labeled): {0: 2560, 1: 2589, 2: 2569}


## Models

In [36]:
# WeightedSAGE:
# GraphSAGE-based model for node classification.
# It stacks multiple SAGEConv layers, applies non-linearity + dropout between them,
# and ends with a linear head for classification.

class WeightedSAGE(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes=NUM_CLASSES, num_layers=2, dropout=0.5):
        super().__init__()

        # Initialize list of GraphSAGE convolution layers
        self.convs = nn.ModuleList()

        # First layer: input -> hidden
        self.convs.append(SAGEConv(in_dim, hidden_dim))

        # Additional hidden layers
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))

        # Final classification head (maps embeddings to class logits)
        self.head = nn.Linear(hidden_dim, num_classes)

        # Dropout probability
        self.dropout = dropout

    def forward(self, x, edge_index, edge_weight=None):
        # Pass through each GraphSAGE layer
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)

            # Apply activation + dropout (except last conv layer)
            if i < len(self.convs) - 1:
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)

        # Final linear layer to produce class scores
        return self.head(x)

In [37]:
# WeightedGAT:
# Graph Attention Network (GAT) for node classification.
# Uses multi-head attention in hidden layers, then a single-head layer,
# followed by a linear classifier.

class WeightedGAT(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes=NUM_CLASSES, num_layers=2, heads=4, dropout=0.5):
        super().__init__()

        # Initialize list of GAT layers
        self.convs = nn.ModuleList()

        # First layer: input -> hidden (multi-head attention)
        self.convs.append(GATConv(in_dim, hidden_dim, heads=heads, dropout=dropout))

        # Middle layers: hidden -> hidden (multi-head)
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(hidden_dim * heads, hidden_dim, heads=heads, dropout=dropout))

        # Final GAT layer: reduce to single head (no concatenation)
        self.convs.append(GATConv(hidden_dim * heads, hidden_dim, heads=1, concat=False, dropout=dropout))

        # Final classification head
        self.head = nn.Linear(hidden_dim, num_classes)

        # Dropout probability
        self.dropout = dropout

    def forward(self, x, edge_index, edge_weight=None):
        # Pass through each GAT layer
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)

            # Apply activation + dropout (except last layer)
            if i < len(self.convs) - 1:
                x = F.elu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)

        # Final linear layer for class logits
        return self.head(x)

## Training loop and eval

In [38]:
# train_one:
# Trains a GNN model using train/validation masks.
# Tracks best validation accuracy and restores best model weights.

def train_one(model, data, train_mask, val_mask, epochs=EPOCHS, lr=LR, verbose=False):
    
    # Move model and data to device (GPU/CPU)
    model = model.to(DEVICE)
    data = data.to(DEVICE)
    train_mask = train_mask.to(DEVICE)
    val_mask = val_mask.to(DEVICE)

    # Initialize optimizer (Adam with weight decay for regularization)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)

    best_val_acc = 0.0
    best_state = None

    # Training loop
    for ep in range(1, epochs + 1):

        # ---- TRAIN STEP ----
        model.train()
        opt.zero_grad()

        # Forward pass: compute logits for all nodes
        logits = model(data.x, data.edge_index, data.edge_weight)

        # Compute loss only on training nodes
        loss = F.cross_entropy(logits[train_mask], data.y[train_mask])

        # Backpropagation
        loss.backward()
        opt.step()

        # ---- VALIDATION STEP ----
        model.eval()
        with torch.no_grad():

            # Forward pass (no gradient tracking)
            logits = model(data.x, data.edge_index, data.edge_weight)
            pred = logits.argmax(dim=-1)

            # Compute validation accuracy
            val_acc = (
                (pred[val_mask] == data.y[val_mask]).float().mean().item()
                if val_mask.any() else 0.0
            )

        # ---- TRACK BEST MODEL ----
        if val_acc > best_val_acc:
            best_val_acc = val_acc

            # Save best model weights (deep copy to CPU)
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

        # ---- LOGGING ----
        if verbose and ep % 25 == 0:
            print(f"  ep {ep:3d}  loss={loss.item():.4f}  val_acc={val_acc:.3f}")

    # Restore best model weights after training
    if best_state is not None:
        model.load_state_dict(best_state)

    return model

In [ ]:
# train_one (Early Stopping Version):
# Trains a GNN model until validation accuracy stops improving.
# Uses patience + min_delta to detect plateau and restores best weights.

PATIENCE = 40       # epochs to wait without improvement
MIN_DELTA = 1e-7    # minimum improvement to reset patience
MAX_EPOCHS = 300    # safety cap


def train_one(model, data, train_mask, val_mask,
              epochs=MAX_EPOCHS, lr=LR, verbose=False,
              patience=PATIENCE, min_delta=MIN_DELTA):

    # ---- SETUP: MOVE TO DEVICE ----
    model = model.to(DEVICE)
    data = data.to(DEVICE)
    train_mask = train_mask.to(DEVICE)
    val_mask = val_mask.to(DEVICE)

    # ---- OPTIMIZER ----
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)

    best_val_acc = 0.0
    best_state = None
    epochs_since_best = 0
    stopped_at = epochs

    # ---- TRAINING LOOP ----
    for ep in range(1, epochs + 1):

        # ---- TRAIN STEP ----
        model.train()
        opt.zero_grad()

        # Forward pass
        logits = model(data.x, data.edge_index, data.edge_weight)

        # Compute loss on training nodes only
        loss = F.cross_entropy(logits[train_mask], data.y[train_mask])

        # Backprop + update
        loss.backward()
        opt.step()

        # ---- VALIDATION STEP ----
        model.eval()
        with torch.no_grad():

            # Forward pass (no gradients)
            logits = model(data.x, data.edge_index, data.edge_weight)
            pred = logits.argmax(dim=-1)

            # Compute validation accuracy
            val_acc = (
                (pred[val_mask] == data.y[val_mask]).float().mean().item()
                if val_mask.any() else 0.0
            )

        # ---- CHECK IMPROVEMENT ----
        if val_acc > best_val_acc + min_delta:
            best_val_acc = val_acc

            # Save best model snapshot
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

            epochs_since_best = 0
        else:
            epochs_since_best += 1

        # ---- LOGGING ----
        if verbose and ep % 25 == 0:
            print(
                f"  ep {ep:3d}  loss={loss.item():.4f}  val_acc={val_acc:.3f}  "
                f"(best={best_val_acc:.3f}, no-improve={epochs_since_best})"
            )

        # ---- EARLY STOP CONDITION ----
        if epochs_since_best >= patience:
            stopped_at = ep
            if verbose:
                print(f"  early stop @ ep {ep}  best_val_acc={best_val_acc:.3f}")
            break

    # ---- RESTORE BEST MODEL ----
    if best_state is not None:
        model.load_state_dict(best_state)

    # ---- ATTACH TRAINING METADATA ----
    model.stopped_at = stopped_at
    model.best_val_acc = best_val_acc

    return model


print(f"early stopping enabled: patience={PATIENCE}, min_delta={MIN_DELTA}, max_epochs={MAX_EPOCHS}")

In [39]:
# eval_subsets:
# Evaluates a trained GNN model on different subsets of nodes:
# - all nodes
# - stock nodes (non-CIK)
# - investor nodes (CIK)
# Returns accuracy and macro-F1 for each subset.

def eval_subsets(model, data, mask):

    # Set model to evaluation mode (disable dropout, etc.)
    model.eval()

    # ---- FORWARD PASS (NO GRADIENTS) ----
    with torch.no_grad():
        logits = model(
            data.x.to(DEVICE),
            data.edge_index.to(DEVICE),
            data.edge_weight.to(DEVICE)
        )

        # Convert logits to predicted class indices
        pred = logits.argmax(dim=-1).cpu()

    # Move ground truth labels to CPU
    y = data.y.cpu()

    out = {}

    # ---- EVALUATE DIFFERENT SUBSETS ----
    for label, sel in [
        ("all", mask),

        # Stock nodes = not CIK (i.e., companies)
        ("stocks", mask & (~data.is_cik.cpu())),

        # Investor nodes = CIK
        ("investors", mask & data.is_cik.cpu()),
    ]:

        # Handle empty subset
        if sel.sum() == 0:
            out[label] = {"n": 0}
            continue

        # Extract true and predicted labels for subset
        yt = y[sel].numpy()
        yp = pred[sel].numpy()

        # Compute metrics
        out[label] = {
            "n": int(sel.sum()),

            # Accuracy: overall correctness
            "accuracy": accuracy_score(yt, yp),

            # Macro-F1: balances performance across classes
            "macro_f1": f1_score(
                yt, yp,
                average="macro",
                labels=[0, 1, 2],
                zero_division=0
            ),
        }

    return out

## Node-mask methodology

Random 20% of labeled nodes held out on the same graph. Random-chance acc = 0.333.

In [18]:
# make_mask_split:
# Splits labeled nodes into train/test masks.
# Randomly selects a fraction (frac) for test, rest for training.

def make_mask_split(data, frac=MASK_FRAC, seed=0):

    # Initialize random generator for reproducibility
    rng = np.random.default_rng(seed)

    # Get indices of nodes that actually have labels
    labeled = data.has_label.cpu().nonzero(as_tuple=True)[0].numpy()

    # Shuffle labeled indices randomly
    perm = rng.permutation(labeled)

    # Compute split point (train vs test)
    cut = int(len(perm) * (1 - frac))

    # Initialize empty boolean masks
    train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)

    # Assign nodes to train and test sets
    train_mask[perm[:cut]] = True
    test_mask[perm[cut:]] = True

    return train_mask, test_mask

In [19]:
# Full pipeline:
# 1. Split labeled nodes into train/test
# 2. Train GraphSAGE and GAT models
# 3. Evaluate both on test set (overall + subsets)
# 4. Compare results in a DataFrame

# ---- CREATE TRAIN / TEST SPLIT ----
train_mask, test_mask = make_mask_split(data)

# Print number of labeled nodes in each split
print(f"train labeled: {int(train_mask.sum()):,}   test labeled: {int(test_mask.sum()):,}")

# ---- TRAIN GRAPH SAGE MODEL ----
sage = WeightedSAGE(
    data.num_node_features,
    HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
)

# Train model (keeps best validation weights)
sage = train_one(sage, data, train_mask, test_mask, verbose=True)

# Evaluate on test set (all / stocks / investors)
res_sage_mask = eval_subsets(sage, data, test_mask)


# ---- TRAIN GAT MODEL ----
gat = WeightedGAT(
    data.num_node_features,
    HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    heads=GAT_HEADS,
    dropout=DROPOUT
)

# Train model
gat = train_one(gat, data, train_mask, test_mask, verbose=True)

# Evaluate on test set
res_gat_mask = eval_subsets(gat, data, test_mask)


# ---- COMPARE RESULTS ----
# Flatten results into a comparison table
results_df = pd.DataFrame({
    "GraphSAGE": {
        f"{k}.{m}": v
        for k, d in res_sage_mask.items()
        for m, v in d.items()
    },
    "GAT": {
        f"{k}.{m}": v
        for k, d in res_gat_mask.items()
        for m, v in d.items()
    },
})

results_df

train labeled: 6,174   test labeled: 1,544
  ep  25  loss=1.0700  val_acc=0.441
  ep  50  loss=1.0510  val_acc=0.448
  ep  75  loss=1.0396  val_acc=0.460
  ep 100  loss=1.0338  val_acc=0.475
  ep 125  loss=1.0289  val_acc=0.478
  ep 150  loss=1.0244  val_acc=0.484
  ep  25  loss=1.0872  val_acc=0.400
  ep  50  loss=1.0794  val_acc=0.414
  ep  75  loss=1.0704  val_acc=0.423
  ep 100  loss=1.0718  val_acc=0.407
  ep 125  loss=1.0710  val_acc=0.431
  ep 150  loss=1.0684  val_acc=0.429


,GraphSAGE,GAT
all.n,1544.000000,1544.000000
all.accuracy,0.485104,0.440415
all.macro_f1,0.461188,0.420231
stocks.n,618.000000,618.000000
stocks.accuracy,0.438511,0.459547
stocks.macro_f1,0.397083,0.431858
investors.n,926.000000,926.000000
investors.accuracy,0.516199,0.427646
investors.macro_f1,0.500127,0.411965


## Two-periods methodology

Train on Δ at (year, quarter−1), test on Δ at (year, quarter).

In [40]:
# two_periods_eval:
# Trains a model on (t-1) quarter and evaluates on (t) quarter.
# This simulates a realistic "predict the future" scenario (no leakage).

def two_periods_eval(year, quarter, model_cls,edges_col_name, **kwargs,):

    # ---- DEFINE PREVIOUS QUARTER (t-1) ----
    py, pq = (year - 1, 4) if quarter == 1 else (year, quarter - 1)

    # ---- BUILD GRAPHS FOR TRAIN (t-1) AND TEST (t) ----
    train_data, _ = build_graph(py, pq,edges_col_name)
    test_data, _ = build_graph(year, quarter,edges_col_name)

    # ---- ALIGN FEATURE DIMENSIONS BETWEEN PERIODS ----
    # (important because feature spaces may differ across quarters)
    Fdim = max(train_data.num_node_features, test_data.num_node_features)

    def pad(d):
        # Pad feature matrix with zeros if needed
        if d.num_node_features < Fdim:
            pad_cols = torch.zeros(d.num_nodes, Fdim - d.num_node_features)
            d.x = torch.cat([d.x, pad_cols], dim=1)
        return d

    train_data = pad(train_data)
    test_data = pad(test_data)

    # ---- INITIALIZE MODEL ----
    model = model_cls(Fdim, HIDDEN_DIM, **kwargs)

    # ---- TRAIN ON PREVIOUS QUARTER ----
    # Note: uses all labeled nodes for training (no validation split)
    model = train_one(
        model,
        train_data,
        train_data.has_label,
        train_data.has_label,
        verbose=False
    )

    # ---- EVALUATE ON NEXT QUARTER ----
    return eval_subsets(model, test_data, test_data.has_label)

In [22]:
res_sage_2p = two_periods_eval(YEAR, QUARTER, WeightedSAGE,EDGES_COLUMN_NAME, num_layers=NUM_LAYERS, dropout=DROPOUT,)
res_gat_2p = two_periods_eval(YEAR, QUARTER, WeightedGAT,EDGES_COLUMN_NAME, num_layers=NUM_LAYERS, heads=GAT_HEADS, dropout=DROPOUT)

pd.DataFrame({
    "GraphSAGE": {f"{k}.{m}": v for k, d in res_sage_2p.items() for m, v in d.items()},
    "GAT": {f"{k}.{m}": v for k, d in res_gat_2p.items() for m, v in d.items()},
})

,GraphSAGE,GAT
all.n,7718.000000,7718.000000
all.accuracy,0.397383,0.306038
all.macro_f1,0.388410,0.193487
stocks.n,2992.000000,2992.000000
stocks.accuracy,0.403075,0.329880
stocks.macro_f1,0.357178,0.182751
investors.n,4726.000000,4726.000000
investors.accuracy,0.393779,0.290944
investors.macro_f1,0.395170,0.195181


## Summary

In [23]:
def flat(res, tag):
    return {
        f"{tag}.all.acc": res["all"].get("accuracy"),
        f"{tag}.all.f1":  res["all"].get("macro_f1"),
        f"{tag}.stocks.acc": res["stocks"].get("accuracy"),
        f"{tag}.investors.acc": res["investors"].get("accuracy"),
    }

summary = pd.DataFrame([
    {"method": "GraphSAGE (mask)",        **flat(res_sage_mask, "test")},
    {"method": "GAT (mask)",              **flat(res_gat_mask,  "test")},
    {"method": "GraphSAGE (two-periods)", **flat(res_sage_2p,   "test")},
    {"method": "GAT (two-periods)",       **flat(res_gat_2p,    "test")},
])
print("Random-chance accuracy = 0.333")
summary

Random-chance accuracy = 0.333


,method,test.all.acc,test.all.f1,test.stocks.acc,test.investors.acc
0,GraphSAGE (mask),0.485104,0.461188,0.438511,0.516199
1,GAT (mask),0.440415,0.420231,0.459547,0.427646
2,GraphSAGE (two-periods),0.397383,0.388410,0.403075,0.393779
3,GAT (two-periods),0.306038,0.193487,0.329880,0.290944


## Notes

- Tertile classification is harder than binary: random-chance acc is 1/3 (0.333) vs 0.5. Anything above ~0.40 is meaningful signal.
- Macro-F1 is reported alongside accuracy because middle-tertile is often the hardest class — pure accuracy can mask middle-class collapse.
- The middle bucket is the hardest because the GNN has to distinguish two soft thresholds rather than one hard sign.
- Bumped `HIDDEN_DIM` to 64 and `EPOCHS` to 150 vs the binary notebook — extra class needs more capacity to separate.

## Per-quarter sweep — two-periods evaluation

Iterates over every (year, quarter) where both Δ-graph (t-1) and Δ-graph (t) exist, runs the two-periods protocol for both SAGE and GAT, and stores the per-quarter results in `quarter_results_df`. The best model across the whole sweep (judged by `test.all.accuracy`) is pickled to `models/` along with its config so it can be reloaded later.

Outputs:
- `quarter_results_df` — one row per quarter with SAGE/GAT accuracy, macro-F1, train time, and early-stop epoch.
- `models/two_periods_quarter_results.csv` — same table on disk.
- `models/best_<tag>_<year>Q<q>.pkl` — pickled state_dict + config of the best model.

In [41]:
# Per-quarter sweep — RESUMABLE + RETRY edition.
#
# Behaviour:
#   * Writes one row to CSV per quarter so we never lose progress.
#   * On rerun, reads the CSV and skips already-completed (year, quarter) pairs.
#   * Wraps the whole sweep in a retry loop: up to 3 attempts.
#       - Between attempts: aggressive GPU cleanup + 5s pause.
#       - If all 3 fail, raises and the notebook stops.
#   * Inner per-model errors (e.g. OOM on GAT for one quarter) are still
#     caught locally and recorded as NaN — they don't abort the sweep.
#
# To start fresh: delete models/two_periods_quarter_results.csv first.
# To resume: just rerun — it skips quarters already in the CSV.

import time
import pickle
import gc
from pathlib import Path

# ---- OUTPUT DIRECTORY ----
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_CSV = MODELS_DIR / "two_periods_quarter_results.csv"


# ---- LIST ALL VIABLE QUARTERS ----
def list_available_quarters():
    """Quarters where both (t-1) and (t+1) exist in TGT_CHANGED_HOLDINGS."""
    yq = handler.query(f"""
        SELECT DISTINCT year, quarter
        FROM {TGT_CHANGED_HOLDINGS}
        WHERE {EDGES_COLUMN_NAME} IS NOT NULL
        ORDER BY year, quarter
    """)
    yq = list(zip(yq["year"].astype(int), yq["quarter"].astype(int)))
    avail = set(yq)
    out = []
    for y, q in yq:
        py, pq = (y - 1, 4) if q == 1 else (y, q - 1)
        ny, nq = next_year_quarter(y, q)
        if (py, pq) in avail and (ny, nq) in avail:
            out.append((y, q))
    return out


# ---- TWO-PERIODS WRAPPER ----
def two_periods_with_model(year, quarter, model_cls, edges_col_name, **kwargs):
    """Returns (model_on_cpu, results, Fdim). Releases GPU memory before returning."""
    py, pq = (year - 1, 4) if quarter == 1 else (year, quarter - 1)
    train_data, _ = build_graph(py, pq, edges_col_name)
    test_data,  _ = build_graph(year, quarter, edges_col_name)

    Fdim = max(train_data.num_node_features, test_data.num_node_features)

    def pad(d):
        if d.num_node_features < Fdim:
            pad_cols = torch.zeros(d.num_nodes, Fdim - d.num_node_features)
            d.x = torch.cat([d.x, pad_cols], dim=1)
        return d

    train_data = pad(train_data)
    test_data  = pad(test_data)

    model = model_cls(Fdim, HIDDEN_DIM, **kwargs)
    model = train_one(model, train_data, train_data.has_label, train_data.has_label, verbose=False)
    results = eval_subsets(model, test_data, test_data.has_label)

    train_data = train_data.cpu()
    test_data  = test_data.cpu()
    model = model.cpu()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return model, results, Fdim


# ---- CSV HELPERS ----
def load_done_set(csv_path):
    """Set of (year, quarter) already in the CSV."""
    if not csv_path.exists():
        return set(), pd.DataFrame()
    df = pd.read_csv(csv_path)
    done = set(zip(df["year"].astype(int), df["quarter"].astype(int)))
    return done, df


def append_row_to_csv(row, csv_path):
    pd.DataFrame([row]).to_csv(
        csv_path, mode="a", header=not csv_path.exists(), index=False)


def aggressive_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


# ---- INNER SWEEP (one attempt) ----
def run_sweep(quarters, csv_path, best):
    done, _ = load_done_set(csv_path)
    remaining = [(y, q) for y, q in quarters if (y, q) not in done]
    print(f"resume: {len(done)} done, {len(remaining)} remaining")

    t_start = time.perf_counter()

    for i, (y, q) in enumerate(remaining, 1):
        row = {"year": y, "quarter": q}

        # ---- SAGE ----
        try:
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            t0 = time.perf_counter()
            m, r, fd = two_periods_with_model(
                y, q, WeightedSAGE, EDGES_COLUMN_NAME,
                num_layers=NUM_LAYERS, dropout=DROPOUT,
            )
            row["sage_train_s"]    = time.perf_counter() - t0
            row["sage_peak_gb"]    = (torch.cuda.max_memory_allocated() / 1e9
                                       if torch.cuda.is_available() else 0.0)
            row["sage_stopped_at"] = getattr(m, "stopped_at", None)
            row["sage_acc"]        = r["all"].get("accuracy")
            row["sage_f1"]         = r["all"].get("macro_f1")
            row["sage_stocks_acc"] = r["stocks"].get("accuracy")
            row["sage_inv_acc"]    = r["investors"].get("accuracy")
            if row["sage_acc"] is not None and row["sage_acc"] > best["acc"]:
                best.update({"acc": row["sage_acc"], "model": m, "tag": "sage",
                             "Fdim": fd, "year": y, "quarter": q})
        except Exception as e:
            print(f"  ! SAGE {y} Q{q} failed: {e.__class__.__name__}: {e}")
        finally:
            for _name in ("m", "r"):
                if _name in dir():
                    try:
                        del globals()[_name]
                    except KeyError:
                        pass
            aggressive_cleanup()

        # ---- GAT ----
        try:
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            t0 = time.perf_counter()
            m, r, fd = two_periods_with_model(
                y, q, WeightedGAT, EDGES_COLUMN_NAME,
                num_layers=NUM_LAYERS, heads=GAT_HEADS, dropout=DROPOUT,
            )
            row["gat_train_s"]    = time.perf_counter() - t0
            row["gat_peak_gb"]    = (torch.cuda.max_memory_allocated() / 1e9
                                      if torch.cuda.is_available() else 0.0)
            row["gat_stopped_at"] = getattr(m, "stopped_at", None)
            row["gat_acc"]        = r["all"].get("accuracy")
            row["gat_f1"]         = r["all"].get("macro_f1")
            row["gat_stocks_acc"] = r["stocks"].get("accuracy")
            row["gat_inv_acc"]    = r["investors"].get("accuracy")
            if row["gat_acc"] is not None and row["gat_acc"] > best["acc"]:
                best.update({"acc": row["gat_acc"], "model": m, "tag": "gat",
                             "Fdim": fd, "year": y, "quarter": q})
        except Exception as e:
            print(f"  ! GAT {y} Q{q} failed: {e.__class__.__name__}: {e}")
        finally:
            for _name in ("m", "r"):
                if _name in dir():
                    try:
                        del globals()[_name]
                    except KeyError:
                        pass
            aggressive_cleanup()

        # ---- INCREMENTAL SAVE ----
        append_row_to_csv(row, csv_path)

        # ---- PROGRESS ----
        elapsed = time.perf_counter() - t_start
        eta = elapsed / i * (len(remaining) - i)
        free_gb = (torch.cuda.mem_get_info()[0] / 1e9
                   if torch.cuda.is_available() else 0.0)
        print(f"  [{len(done) + i:2d}/{len(quarters)}] {y} Q{q}  "
              f"SAGE={row.get('sage_acc', float('nan')):.3f} "
              f"(peak {row.get('sage_peak_gb', 0):.2f}GB)  "
              f"GAT={row.get('gat_acc', float('nan')):.3f} "
              f"(peak {row.get('gat_peak_gb', 0):.2f}GB)  "
              f"free {free_gb:.2f}GB  ETA {eta/60:.1f}m")


# ---- OUTER RETRY LOOP ----
quarters = list_available_quarters()
print(f"running two-periods on {len(quarters)} quarters: {quarters[0]} … {quarters[-1]}")

MAX_ATTEMPTS = 3
best = {"acc": -1.0, "model": None, "tag": None, "Fdim": None, "year": None, "quarter": None}
overall_start = time.perf_counter()
last_error = None

for attempt in range(1, MAX_ATTEMPTS + 1):
    print(f"\n=== attempt {attempt}/{MAX_ATTEMPTS} ===")
    try:
        run_sweep(quarters, RESULTS_CSV, best)
        last_error = None
        done_now, _ = load_done_set(RESULTS_CSV)
        if len(done_now) >= len(quarters):
            print("all quarters complete.")
            break
    except KeyboardInterrupt:
        raise
    except Exception as e:
        last_error = e
        print(f"!! attempt {attempt} crashed: {e.__class__.__name__}: {e}")
        aggressive_cleanup()
        time.sleep(5)
        continue

if last_error is not None:
    raise RuntimeError(
        f"sweep failed after {MAX_ATTEMPTS} attempts; last error: "
        f"{last_error.__class__.__name__}: {last_error}"
    )

# ---- FINALISE ----
quarter_results_df = pd.read_csv(RESULTS_CSV)
print(f"\nfinished in {(time.perf_counter() - overall_start)/60:.1f} min")
print(f"best model: {best['tag']} on {best['year']} Q{best['quarter']}  (acc={best['acc']:.3f})")


# ---- SAVE BEST MODEL ----
if best["model"] is not None:
    best_path = MODELS_DIR / f"best_{best['tag']}_{best['year']}Q{best['quarter']}.pkl"
    with open(best_path, "wb") as f:
        pickle.dump({
            "tag": best["tag"],
            "year": best["year"],
            "quarter": best["quarter"],
            "Fdim": best["Fdim"],
            "test_acc": best["acc"],
            "state_dict": {k: v.cpu() for k, v in best["model"].state_dict().items()},
            "config": {
                "HIDDEN_DIM": HIDDEN_DIM,
                "NUM_LAYERS": NUM_LAYERS,
                "GAT_HEADS": GAT_HEADS,
                "DROPOUT": DROPOUT,
                "NUM_CLASSES": NUM_CLASSES,
                "EDGES_COLUMN_NAME": EDGES_COLUMN_NAME,
            },
        }, f)
    print(f"saved → {best_path}")
else:
    print("no best model captured this run (only resumed previously-saved rows).")

print(f"results CSV → {RESULTS_CSV}")
quarter_results_df

running two-periods on 46 quarters: (2013, 4) … (2025, 1)

=== attempt 1/3 ===
resume: 0 done, 46 remaining
  [ 1/46] 2013 Q4  SAGE=0.334 (peak 0.08GB)  GAT=0.341 (peak 0.21GB)  free 5.23GB  ETA 13.9m
  [ 2/46] 2014 Q1  SAGE=0.362 (peak 0.42GB)  GAT=0.336 (peak 2.79GB)  free 5.23GB  ETA 12.7m
  [ 3/46] 2014 Q2  SAGE=0.435 (peak 0.45GB)  GAT=0.382 (peak 5.52GB)  free 5.23GB  ETA 24.6m
  [ 4/46] 2014 Q3  SAGE=0.345 (peak 0.46GB)  GAT=0.358 (peak 5.67GB)  free 5.23GB  ETA 31.6m
  [ 5/46] 2014 Q4  SAGE=0.384 (peak 0.47GB)  GAT=0.306 (peak 5.79GB)  free 5.23GB  ETA 35.3m
  [ 6/46] 2015 Q1  SAGE=0.394 (peak 0.49GB)  GAT=0.360 (peak 5.85GB)  free 5.23GB  ETA 37.0m
  [ 7/46] 2015 Q2  SAGE=0.376 (peak 0.49GB)  GAT=0.388 (peak 6.16GB)  free 5.23GB  ETA 38.3m
  [ 8/46] 2015 Q3  SAGE=0.438 (peak 0.50GB)  GAT=0.396 (peak 6.22GB)  free 5.23GB  ETA 39.2m
  [ 9/46] 2015 Q4  SAGE=0.415 (peak 0.49GB)  GAT=0.386 (peak 6.26GB)  free 5.23GB  ETA 39.5m
  [10/46] 2016 Q1  SAGE=0.413 (peak 0.50GB)  GAT=0.335 

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Aggregate summary across all quarters:
# Means / stds of accuracy, macro-F1, train time, and early-stop epoch.

acc_cols  = ["sage_acc", "gat_acc"]
f1_cols   = ["sage_f1",  "gat_f1"]
time_cols = ["sage_train_s", "gat_train_s"]

print(f"quarters run: {len(quarter_results_df)}   |   random-chance acc = {1/NUM_CLASSES:.3f}\n")

# ---- ACCURACY (mean ± std) ----
print("accuracy  mean ± std:")
print(quarter_results_df[acc_cols].agg(["mean", "std"]).T.round(3))

# ---- MACRO-F1 (mean ± std) ----
print("\nmacro-F1  mean ± std:")
print(quarter_results_df[f1_cols].agg(["mean", "std"]).T.round(3))

# ---- TRAINING TIME (s) ----
print("\ntrain time (s) mean:")
print(quarter_results_df[time_cols].mean().round(2))

# ---- EARLY-STOP EPOCH ----
print("\nearly-stop epoch mean:")
print(quarter_results_df[["sage_stopped_at", "gat_stopped_at"]].mean().round(1))

# ---- TOP-5 QUARTERS BY BEST MODEL ----
print("\ntop-5 quarters by best (max(SAGE, GAT)) accuracy:")
quarter_results_df.assign(best=quarter_results_df[acc_cols].max(axis=1)) \
    .sort_values("best", ascending=False) \
    .head(5)[["year", "quarter", "sage_acc", "gat_acc", "best"]]